# 12.20 - LangChain Synthesis & Capstone

**Phase:** 12 - LangChain

**Status:** VERIFIED

---

## 1. What Are We Solving?

Demonstrate mastery of LangChain by building a complete application.

## 2. Why Does This Matter?

The capstone proves you can build real LLM applications with LangChain.

## 3. Prerequisites

- All previous notebooks in Phase 12

## 4. Learning Objectives

- Build a complete RAG application
- Use chains, memory, and tools
- Evaluate and deploy

## 5. Mental Model

Complete app = data loading + processing + retrieval + generation + evaluation.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
print("Libraries loaded.")

Libraries loaded.


## 6. Project: Knowledge Base Q&A

In [2]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

# Knowledge base
docs = [
    Document(page_content="Python is a high-level programming language. It was created by Guido van Rossum and released in 1991. Python emphasizes code readability and supports multiple programming paradigms.", metadata={"source": "python"}),
    Document(page_content="Machine learning is a subset of artificial intelligence. It enables systems to learn from data. Common types include supervised, unsupervised, and reinforcement learning.", metadata={"source": "ml"}),
    Document(page_content="Deep learning is a subset of machine learning using neural networks. It excels at image recognition, natural language processing, and speech recognition.", metadata={"source": "dl"}),
    Document(page_content="LangChain is a framework for building LLM applications. It provides tools for prompt management, chaining, memory, retrieval, and agents.", metadata={"source": "langchain"}),
    Document(page_content="RAG combines retrieval with generation. It retrieves relevant documents and uses them as context for LLM answers, reducing hallucinations.", metadata={"source": "rag"}),
]

print("Knowledge base: " + str(len(docs)) + " documents")

Knowledge base: 5 documents


## 7. Retrieval System

In [3]:
def retrieve(query, docs, k=2):
    scores = []
    query_words = set(query.lower().split())
    for doc in docs:
        doc_words = set(doc.page_content.lower().split())
        overlap = len(query_words & doc_words)
        scores.append((doc, overlap))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scores[:k]]

## 8. Generation Chain

In [4]:
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Use the context to answer the question.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

## 9. Complete RAG Application

In [5]:
def knowledge_qa(question):
    # Retrieve
    retrieved = retrieve(question, docs)
    context = "\n".join(d.page_content for d in retrieved)
    sources = [d.metadata["source"] for d in retrieved]
    
    # Generate
    answer = chain.invoke({"context": context, "question": question})
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context_length": len(context)
    }

# Test
questions = [
    "What is Python?",
    "How does machine learning work?",
    "What is RAG?",
    "What is LangChain used for?"
]

for q in questions:
    result = knowledge_qa(q)
    print("Q:", result["question"])
    print("A:", result["answer"][:150])
    print("Sources:", result["sources"])
    print()

Q: What is Python?
A: Python is a high-level programming language created by Guido van Rossum and released in 1991. It emphasizes code readability and supports multiple pro
Sources: ['python', 'ml']



Q: How does machine learning work?
A: Based on the provided context, machine learning works by enabling systems to learn from data. It is a subset of artificial intelligence that includes 
Sources: ['ml', 'dl']



Q: What is RAG?
A: The provided context does not contain information about RAG. The context only defines Python and Machine Learning.
Sources: ['python', 'ml']



Q: What is LangChain used for?
A: LangChain is a framework used for building LLM applications. It provides tools for prompt management, chaining, memory, retrieval, and agents.
Sources: ['langchain', 'python']



## 10. Evaluation

In [6]:
def evaluate(questions, ground_truth):
    correct = 0
    for q, expected in zip(questions, ground_truth):
        result = knowledge_qa(q)
        answer_lower = result["answer"].lower()
        if any(word in answer_lower for word in expected.lower().split()):
            correct += 1
    return correct / len(questions)

ground_truth = [
    "guido van rossum",
    "learns from data",
    "retrieval",
    "llm applications"
]

accuracy = evaluate(questions, ground_truth)
print("Evaluation accuracy:", round(accuracy, 4))

Evaluation accuracy: 0.75


## 11. Project Ideas

1. Document Q&A bot
2. Code review assistant
3. Research paper summarizer
4. Customer support bot
5. Knowledge base chatbot

## 12. Summary

You built a complete RAG application: knowledge base, retrieval, generation, and evaluation. This is the foundation for real LLM products.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [langchain, langchain-groq]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```